# ViT-B/16 with detail skips on the EfficientViT BTXRD split

The convolutional decoder fuses the 14×14 ViT features with image features at
28×28, 56×56 and 112×112. Initialize the encoder from ImageNet-1K only.
Use the same positive training images, normalization, flips, rotations, loss,
50 epochs, batch size and held-out scoring as qunthy/efficientvit-seg.
Select the checkpoint on validation micro Dice at 0.5. Apply the fixed
original/flip ensemble and threshold 0.5 once on test after selection.

In [ ]:
import os
import json
import random
import time
from pathlib import Path

os.environ['HF_HUB_CACHE'] = '/kaggle/temp/huggingface/hub'
import numpy as np
import torch
import torchvision
from huggingface_hub import hf_hub_download
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

torch.hub.set_dir('/kaggle/temp/torch')
assert torch.cuda.is_available(), 'Kaggle GPU required'
device = torch.device('cuda')
random.seed(42)
torch.manual_seed(42)
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(), flush=True)

In [ ]:
import hashlib

DATASET_REVISION = 'cf8b0569b18e43e7cecab412a06e60aeaf258b32'
EXPECTED_SHA256 = '59ce94244fdf997f0c68e6a378690a4359e6c411f7fda7ac2f3994489637884b'
archive = Path(hf_hub_download(repo_id='torusama/xla-seminar-uia', repo_type='dataset',
                               filename='btxrd_final.zip', revision=DATASET_REVISION))
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(f'Dataset checksum mismatch: {digest}')
print('BTXRD downloaded; SHA-256:', digest, flush=True)

In [ ]:
"""Read the pinned Hugging Face archive and rasterize polygon masks."""
import hashlib
import io
import json
import os
from pathlib import Path
from zipfile import ZipFile

import torch
from PIL import Image, ImageDraw
from torch.utils.data import Dataset
from torchvision.transforms import functional as TF


class PolygonArchive(Dataset):
    """Binary union of all annotated polygons; normal samples have empty masks."""

    def __init__(self, archive: Path, split: str, excluded_hashes: set[str] | None = None):
        self.archive = Path(archive)
        self.samples = []
        self.excluded = 0
        self._zip = None
        self._zip_pid = None
        excluded_hashes = excluded_hashes or set()
        with ZipFile(self.archive) as z:
            names = set(z.namelist())
            annotations = sorted(n for n in names if n.startswith(f"{split}/annotations/")
                                 and n.endswith(".json"))
            for name in annotations:
                annotation = json.loads(z.read(name))
                category = Path(name).parent.name
                image_name = f"{split}/images/{category}/{annotation['image_name']}"
                if image_name not in names:
                    raise ValueError(f"Missing image for {name}: {image_name}")
                digest = hashlib.sha256(z.read(image_name)).hexdigest()
                if digest in excluded_hashes:
                    self.excluded += 1
                    continue
                self.samples.append((image_name, name, digest))
        if not self.samples:
            raise ValueError(f"No samples in {self.archive} split {split}")

    @property
    def hashes(self) -> set[str]:
        return {sample[2] for sample in self.samples}

    def __len__(self):
        return len(self.samples)

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_zip"] = None
        state["_zip_pid"] = None
        return state

    def __getitem__(self, index):
        image_name, annotation_name, _ = self.samples[index]
        if self._zip is None or self._zip_pid != os.getpid():
            if self._zip is not None:
                self._zip.close()
            self._zip = ZipFile(self.archive)
            self._zip_pid = os.getpid()
        annotation = json.loads(self._zip.read(annotation_name))
        with Image.open(io.BytesIO(self._zip.read(image_name))) as source:
            image = source.convert("RGB")
        expected = (annotation["image_width"], annotation["image_height"])
        if image.size != expected or expected != (224, 224):
            raise ValueError(f"Image and annotation dimensions do not match 224x224: {image_name}")
        mask = Image.new("L", image.size, 0)
        draw = ImageDraw.Draw(mask)
        count = 0
        for item in annotation["annotations"]:
            for polygon in item.get("polygons", []):
                if len(polygon) < 3:
                    raise ValueError(f"Invalid polygon: {annotation_name}")
                draw.polygon([tuple(point) for point in polygon], fill=1)
                count += 1
        if not count and annotation["class_label"].lower() != "normal":
            raise ValueError(f"Non-normal sample has no polygons: {annotation_name}")
        pixels = TF.normalize(TF.to_tensor(image), mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        target = torch.frombuffer(bytearray(mask.tobytes()), dtype=torch.uint8).reshape(1, 224, 224)
        return pixels, target.float()


def make_splits(archive: Path):
    """Remove conflicting labels and train duplicates of held-out images."""
    splits = [PolygonArchive(archive, name) for name in ("train", "val", "test")]
    categories = {}
    for split in splits:
        for image_name, _, digest in split.samples:
            categories.setdefault(digest, set()).add(Path(image_name).parent.name)
    conflicts = {digest for digest, labels in categories.items() if len(labels) > 1}
    for split in splits:
        original = len(split.samples)
        split.samples = [sample for sample in split.samples if sample[2] not in conflicts]
        split.excluded = original - len(split.samples)
    train, val, test = splits
    if val.hashes & test.hashes:
        raise ValueError("Validation and test contain identical images; resolve the split first")
    held_out = val.hashes | test.hashes
    original = len(train.samples)
    train.samples = [sample for sample in train.samples if sample[2] not in held_out]
    train.excluded += original - len(train.samples)
    if any(not split.samples for split in splits):
        raise ValueError("A split is empty after removing conflicting and duplicate images")
    return train, val, test

In [ ]:
class TrainAugmentDataset(Dataset):
    def __init__(self, base):
        self.base = base
        self.indices = [i for i in range(len(base)) if base[i][1].sum().item() > 0]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, mask = self.base[self.indices[idx]]
        if random.random() < 0.5:
            image = torch.flip(image, dims=[-1])
            mask = torch.flip(mask, dims=[-1])
        if random.random() < 0.5:
            angle = random.uniform(-15.0, 15.0)
            image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
        return image, mask

In [ ]:
train_set, val_set, test_set = make_splits(archive)
train_dataset = TrainAugmentDataset(train_set)
samples = {
    'train': [train_set.samples[i] for i in train_dataset.indices],
    'val': val_set.samples,
    'test': test_set.samples,
}
expected = {
    'train': (1302, 'd0c6d06e7bec9f37e2dcbdd18231e028b16759231e8cb6c654016d51f58f5db6'),
    'val': (562, '2b2f1d62214c72f4cc7ca8c8bf668bc7153ff7f92b1889f90c1124c187b18a61'),
    'test': (561, 'e151e8391d3d7025b674a0b5d8faf832362f44c7e446c9d526dd9f85305a4c8f'),
}
manifest = {}
for split, rows in samples.items():
    content = ''.join('\t'.join(row) + '\n' for row in rows)
    signature = hashlib.sha256(content.encode()).hexdigest()
    assert (len(rows), signature) == expected[split], f'{split} membership differs from EViT'
    manifest[split] = {'count': len(rows), 'sha256': signature}
print('Verified EViT split:', manifest, flush=True)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=8, num_workers=2)
test_loader = DataLoader(test_set, batch_size=8, num_workers=2)

In [ ]:
"""ViT-B/16 segmentation with convolutional skip features from the input image."""
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import ViT_B_16_Weights, vit_b_16


def conv_block(inputs, outputs, stride=1):
    return nn.Sequential(
        nn.Conv2d(inputs, outputs, 3, stride=stride, padding=1, bias=False),
        nn.GroupNorm(8, outputs),
        nn.GELU(),
    )


def upsample(features, reference):
    return F.interpolate(features, size=reference.shape[-2:], mode="bilinear", align_corners=False)


class DetailViT(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = vit_b_16(weights=weights)
        self.backbone.heads = nn.Identity()
        self.half_detail = conv_block(3, 24, stride=2)
        self.quarter_detail = conv_block(24, 48, stride=2)
        self.eighth_detail = conv_block(48, 64, stride=2)
        self.context = nn.Conv2d(self.backbone.hidden_dim, 128, 1)
        self.decode_eighth = conv_block(128 + 64, 96)
        self.decode_quarter = conv_block(96 + 48, 64)
        self.decode_half = conv_block(64 + 24, 32)
        self.mask = nn.Conv2d(32, 1, 1)

    def forward(self, image):
        patches = self.backbone.conv_proj(image)
        tokens = patches.flatten(2).transpose(1, 2)
        cls = self.backbone.class_token.expand(image.size(0), -1, -1)
        encoded = self.backbone.encoder(torch.cat((cls, tokens), dim=1))[:, 1:]
        context = self.context(encoded.transpose(1, 2).reshape_as(patches))
        half = self.half_detail(image)
        quarter = self.quarter_detail(half)
        eighth = self.eighth_detail(quarter)
        x = self.decode_eighth(torch.cat((upsample(context, eighth), eighth), dim=1))
        x = self.decode_quarter(torch.cat((upsample(x, quarter), quarter), dim=1))
        x = self.decode_half(torch.cat((upsample(x, half), half), dim=1))
        return upsample(self.mask(x), image)

In [ ]:
"""Training and evaluation utilities for binary ViT segmentation."""
import torch
from torch.nn import functional as F


def segmentation_loss(logits, target, positive_weight=10.0, dice_reduction="batch", small_dice_weight=0.0):
    if dice_reduction not in ("batch", "image"):
        raise ValueError("dice_reduction must be batch or image")
    probabilities = logits.sigmoid()
    if positive_weight is None:
        intersection = (probabilities * target).sum(dim=(1, 2, 3))
        denominator = probabilities.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        return F.binary_cross_entropy_with_logits(logits, target) + (
            1 - ((2 * intersection + 1) / (denominator + 1)).mean()
        )
    dimensions = (1, 2, 3) if dice_reduction == "image" else (0, 1, 2, 3)
    intersection = (probabilities * target).sum(dim=dimensions)
    denominator = probabilities.sum(dim=dimensions) + target.sum(dim=dimensions)
    dice_loss = 1 - ((2 * intersection + 1) / (denominator + 1)).mean()
    weight = logits.new_tensor(positive_weight)
    loss = F.binary_cross_entropy_with_logits(logits, target, pos_weight=weight) + dice_loss
    if small_dice_weight:
        areas = target.sum(dim=(1, 2, 3))
        selected = (areas > 0) & (areas < 256)
        if selected.any():
            small_probs, small_targets = probabilities[selected], target[selected]
            intersection = (small_probs * small_targets).sum(dim=(1, 2, 3))
            denominator = small_probs.sum(dim=(1, 2, 3)) + small_targets.sum(dim=(1, 2, 3))
            loss = loss + small_dice_weight * (1 - (2 * intersection + 1) / (denominator + 1)).mean()
    return loss


def run_epoch(model, loader, device, optimizer=None, positive_weight=10.0,
              dice_reduction="batch", small_dice_weight=0.0):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    tp = fp = fn = 0
    count = 0
    with torch.set_grad_enabled(training):
        for images, target in loader:
            images, target = images.to(device), target.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = segmentation_loss(logits, target, positive_weight, dice_reduction, small_dice_weight)
            if training:
                loss.backward()
                optimizer.step()
            batch = images.shape[0]
            total_loss += loss.item() * batch
            count += batch
            prediction = logits.detach() >= 0
            truth = target.bool()
            tp += (prediction & truth).sum().item()
            fp += (prediction & ~truth).sum().item()
            fn += (~prediction & truth).sum().item()
    return {"loss": total_loss / count, "dice": (2 * tp / (2 * tp + fp + fn)) if tp + fp + fn else 1.0,
            "iou": (tp / (tp + fp + fn)) if tp + fp + fn else 1.0}

In [ ]:
output = Path('/kaggle/working')
model = DetailViT(pretrained=True).to(device)
head = [p for name, p in model.named_parameters() if not name.startswith('backbone.')]
optimizer = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': 2e-5},
                               {'params': head, 'lr': 2e-4}], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)
checkpoint_path = output / 'best_vit_detail.pt'
history = []
best = -1.0
for epoch in range(1, 51):
    train = run_epoch(model, train_loader, device, optimizer, positive_weight=10)
    val = run_epoch(model, val_loader, device, positive_weight=10)
    history.append({'epoch': epoch, 'train': train, 'val': val, 'lr': scheduler.get_last_lr()})
    print(json.dumps(history[-1]), flush=True)
    if val['dice'] > best:
        best = val['dice']
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'validation': val,
                    'archive_sha256': digest, 'split_manifest': manifest,
                    'architecture': 'DetailViT', 'pretraining': 'ViT_B_16_Weights.IMAGENET1K_V1'},
                   checkpoint_path)
    scheduler.step()
    (output / 'history.json').write_text(json.dumps(history, indent=2))

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint['model'], strict=True)
model.eval()
def predict_tta(images):
    with torch.no_grad():
        prob_original = model(images).sigmoid()
        flipped_input = torch.flip(images, dims=[3])
        prob_flipped = torch.flip(model(flipped_input), dims=[3]).sigmoid()
    return (prob_original + prob_flipped) / 2.0

tp = fp = fn = 0
macro_dices = []
macro_ious = []
normal_total = 0
normal_clean = 0
inference_ms = []

with torch.no_grad():
    for images, target in test_loader:
        images = images.to(device)
        target = target.to(device)

        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        prob = predict_tta(images)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        inference_ms.append(elapsed / images.shape[0] * 1000.0)

        pred = prob >= 0.5
        truth = target.bool()

        tp += (pred & truth).sum().item()
        fp += (pred & ~truth).sum().item()
        fn += (~pred & truth).sum().item()

        for i in range(images.shape[0]):
            pred_i = pred[i]
            truth_i = truth[i]
            if target[i].sum().item() > 0:
                intersection = (pred_i & truth_i).sum().item()
                pred_pixels = pred_i.sum().item()
                truth_pixels = truth_i.sum().item()
                union = (pred_i | truth_i).sum().item()
                dice = (2.0 * intersection + 1e-6) / (pred_pixels + truth_pixels + 1e-6)
                iou = (intersection + 1e-6) / (union + 1e-6)
                macro_dices.append(dice)
                macro_ious.append(iou)
            else:
                normal_total += 1
                if pred[i].sum().item() < 25:
                    normal_clean += 1

global_dice = (2 * tp / (2 * tp + fp + fn)) if (tp + fp + fn) else 1.0
global_iou = (tp / (tp + fp + fn)) if (tp + fp + fn) else 1.0
mean_dice = float(np.mean(macro_dices)) if macro_dices else 1.0
mean_iou = float(np.mean(macro_ious)) if macro_ious else 1.0
specificity = (normal_clean / normal_total) if normal_total else 1.0

params_m = sum(p.numel() for p in model.parameters()) / 1e6
checkpoint_size_mb = checkpoint_path.stat().st_size / (1024 * 1024)
avg_inference_ms = float(np.mean(inference_ms))

metrics = {
    'global_micro_dice': global_dice,
    'global_micro_iou': global_iou,
    'mean_macro_dice_tumor': mean_dice,
    'mean_macro_iou_tumor': mean_iou,
    'specificity_normal': specificity,
    'parameters_m': params_m,
    'checkpoint_size_mb': checkpoint_size_mb,
    'avg_inference_ms_per_image': avg_inference_ms,
    'tumor_test_images': len(macro_dices),
    'normal_test_images': normal_total,
}
print(json.dumps(metrics, indent=2), flush=True)
Path('/kaggle/working/vit_detail_test_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

In [ ]:
# Published best-checkpoint test output of qunthy/efficientvit-seg
evit = {'global_micro_dice': 0.7403773162089105, 'global_micro_iou': 0.5877770587463502,
        'mean_macro_dice_tumor': 0.5708889301999, 'mean_macro_iou_tumor': 0.46695038501914377,
        'specificity_normal': 0.6903914590747331}
assert (metrics['tumor_test_images'], metrics['normal_test_images']) == (280, 281)
comparison = {'reference': 'qunthy/efficientvit-seg', 'archive_sha256': digest,
              'split_manifest': manifest, 'selected_epoch': checkpoint['epoch'],
              'validation_dice': checkpoint['validation']['dice'], 'threshold': 0.5,
              'test_tta': 'mean of original and unflipped horizontal-flip probabilities',
              'evit_test': evit, 'vit_detail_test': metrics,
              'vit_minus_evit_test': {key: metrics[key] - value for key, value in evit.items()}}
(output / 'comparison.json').write_text(json.dumps(comparison, indent=2))
print(json.dumps(comparison, indent=2))